# Shortest Path and Traveling Salesman (Public Transport Stops) Problem
### Author: Jakub Stępkowski

Algorithms used:
- Dijkstra's algorithm
- A* algorithm
- Tabu Search

## All imports

In [24]:
import os
import random
import heapq
import pickle
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
from functools import total_ordering
from typing import Callable
from math import atan2, cos, radians, sin, sqrt
from copy import deepcopy
from typing import Sequence

# Shortest Path

## Sample data

In [6]:
def load_df(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    return df

raw_df = load_df("data/wroclaw-mpk.csv")

print("Data Loaded\n")
print(raw_df.head())

print("\nData Types\n")
print(raw_df.info())

print("\nDescription\n")
print(raw_df.describe())

print("\nMarginal departure_times\n")
print(raw_df["departure_time"].max(), raw_df["departure_time"].min(), "\n")

print("\nMarginal arrival_time\n")
print(raw_df["arrival_time"].max(), raw_df["arrival_time"].min(), "\n")

Data Loaded

   Unnamed: 0       company line departure_time arrival_time  \
0           0  MPK Autobusy    A       20:52:00     20:53:00   
1           1  MPK Autobusy    A       20:53:00     20:54:00   
2           2  MPK Autobusy    A       20:54:00     20:55:00   
3           3  MPK Autobusy    A       20:55:00     20:57:00   
4           4  MPK Autobusy    A       20:57:00     20:59:00   

             start_stop              end_stop  start_stop_lat  start_stop_lon  \
0   Zajezdnia Obornicka              Paprotna       51.148737       17.021069   
1              Paprotna  Obornicka (Wołowska)       51.147752       17.020539   
2  Obornicka (Wołowska)            Bezpieczna       51.144385       17.023735   
3            Bezpieczna              Bałtycka       51.141360       17.026376   
4              Bałtycka         Broniewskiego       51.136632       17.030617   

   end_stop_lat  end_stop_lon  
0     51.147752     17.020539  
1     51.144385     17.023735  
2     51.141360    

## Graph implementation
The structure was implemented for easier access to the data and to avoid mistakes.
Models store and manage the data.

In [7]:
@total_ordering
@dataclass
class Edge:
    start_stop_name: str
    end_stop_name: str
    departure_sec: int
    arrival_sec: int
    line: str
    company: str

    @property
    def duration(self) -> int:
        return self.arrival_sec - self.departure_sec

    def __eq__(self, other):
        if not isinstance(other, Edge):
            return NotImplemented

        return (
            self.start_stop_name == other.start_stop_name
            and self.end_stop_name == other.end_stop_name
            and self.departure_sec == other.departure_sec
            and self.arrival_sec == other.arrival_sec
            and self.line == other.line
            and self.company == other.company
        )

    def __lt__(self, other):
        if not isinstance(other, Edge):
            return NotImplemented

        return (
            [
                self.start_stop_name,
                self.end_stop_name,
                self.departure_sec,
                self.arrival_sec,
                self.line,
                self.company,
            ] < [
                other.start_stop_name,
                other.end_stop_name,
                other.departure_sec,
                other.arrival_sec,
                other.line,
                other.company,
            ]
        )

In [8]:
@dataclass
class Node:
    """
    One Node for specific stop name.
    Many variants of the same stop have averaged coordinates.
    """
    stop_name: str
    latitude: float
    longitude: float
    outgoing_edges: list[Edge] = field(default_factory=list)

In [9]:
class Graph:

    def __init__(self, nodes: dict[str, Node]):
        self.nodes = nodes

        # Sort outgoing edges for each node by departure time
        for node in nodes.values():
            node.outgoing_edges = sorted(node.outgoing_edges, key=lambda x: x.departure_sec)

    @staticmethod
    def _normalize_df(df: pd.DataFrame) -> pd.DataFrame:
        seconds_in_a_day = 60 * 60 * 24

        df["line"] = df["line"].astype(str).str.strip().str.lower()
        df["company"] = df["company"].astype(str).str.strip().str.lower()

        df["departure_sec"] = pd.to_timedelta(df["departure_time"]).dt.total_seconds() % seconds_in_a_day
        df["arrival_sec"] = pd.to_timedelta(df["arrival_time"]).dt.total_seconds() % seconds_in_a_day

        df["start_stop"] = df["start_stop"].astype(str).str.strip().str.lower()
        df["end_stop"] = df["end_stop"].astype(str).str.strip().str.lower()

        return df

    @classmethod
    def create_from_df(cls, df: pd.DataFrame):
        df = cls._normalize_df(df)

        nodes_raw = defaultdict(set)
        for _, row in df.iterrows():
            nodes_raw[row['start_stop']].add((row['start_stop_lat'], row['start_stop_lon']))
            nodes_raw[row['end_stop']].add((row['end_stop_lat'], row['end_stop_lon']))

        nodes_avg: dict[str, Node] = {}
        for stop_name, coords_set in nodes_raw.items():
            nodes_avg[stop_name] = Node(
                stop_name=stop_name,
                latitude=sum([c[0] for c in coords_set]) / len(coords_set),
                longitude=sum([c[1] for c in coords_set]) / len(coords_set)
            )

        for _, row in df.iterrows():
            nodes_avg[row['start_stop']].outgoing_edges.append(
                Edge(
                    start_stop_name=row['start_stop'],
                    end_stop_name=row['end_stop'],
                    departure_sec=row['departure_sec'],
                    arrival_sec=row['arrival_sec'],
                    line=row['line'],
                    company=row['company']
                )
            )

        return cls(nodes=nodes_avg)


## Distance functions
Haversine distance give as more accurate results than Euclidean distance because of the Earth curvature.

In [10]:
def euclides_distance(graph: Graph, stop: str, end_stop: str):
    lat1 = graph.nodes[stop].latitude
    lon1 = graph.nodes[stop].longitude
    lat2 = graph.nodes[end_stop].latitude
    lon2 = graph.nodes[end_stop].longitude

    distance = ((lat1 - lat2) ** 2 + (lon1 - lon2) ** 2) ** 0.5
    return distance


def haversine_distance(graph: Graph, stop: str, end_stop: str) -> float:
    # Radius of the Earth in kilometers
    earth_radius = 6371.0

    # Convert latitude and longitude from degrees to radians
    lat_1 = radians(graph.nodes[stop].latitude)
    lon_1 = radians(graph.nodes[stop].longitude)
    lat_2 = radians(graph.nodes[end_stop].latitude)
    lon_2 = radians(graph.nodes[end_stop].longitude)

    # Differences
    d_lat = lat_2 - lat_1
    d_lon = lon_2 - lon_1

    # Haversine formula
    a = sin(d_lat / 2)**2 + cos(lat_1) * cos(lat_2) * sin(d_lon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    # Distance in kilometers
    return earth_radius * c


## Algorithms

### Dijkstra's algorithm

In [11]:
SECONDS_IN_DAY = 24 * 3600

def dijkstra_shortest_travel_time(
    graph: Graph, start_stop: str, end_stop: str, start_time_sec: int
) -> tuple[list[Edge] | None, int, int]:
    # Priority queue (start_time, start_stop, last_line, path, n_lines)
    queue = [(start_time_sec, start_stop, None, [], 0)]

    # The best arrival times to each stop
    best_arrival_times = {stop: float('inf') for stop in graph.nodes}
    best_arrival_times[start_stop] = start_time_sec
    visited = set()

    while queue:
        # Pop the stop with the earliest arrival time
        current_time, current_stop, last_line, path, n_lines = heapq.heappop(queue)

        # Check if we have already visited this stop
        if current_stop in visited:
            continue
        visited.add(current_stop)

        # End algorithm if we are at the destination
        if current_stop == end_stop:
            return path, current_time, n_lines

        # Get current node and iterate over all outgoing edges
        current_node = graph.nodes[current_stop]
        for edge in current_node.outgoing_edges:

            # Add one day if we are after midnight
            arrival_time = edge.arrival_sec + ((current_time // SECONDS_IN_DAY) * SECONDS_IN_DAY)

            # We cannot go earlier than we arrived at current stop
            if edge.departure_sec < current_time % SECONDS_IN_DAY:
                arrival_time = arrival_time + SECONDS_IN_DAY

            # If departure time is after arrival time, we know it's midnight case
            if edge.departure_sec > edge.arrival_sec:
                arrival_time = arrival_time + SECONDS_IN_DAY

            # Update the number of routes if we changed the line
            n_lines_new = n_lines
            if last_line != edge.line or current_time % SECONDS_IN_DAY != edge.departure_sec:
                n_lines_new += 1

            # Update best time and push queue if we found a better path
            if arrival_time < best_arrival_times[edge.end_stop_name]:
                best_arrival_times[edge.end_stop_name] = arrival_time
                new_path = path + [edge]
                heapq.heappush(
                    queue,
                    (arrival_time, edge.end_stop_name, edge.line, new_path, n_lines_new)
                )

    return None, -1, -1  # No path found

### A* algorithm
heuristic function as argument, extra line changes cost

In [12]:
def astar_shortest_travel(
    graph: Graph,
    start_stop: str,
    end_stop: str,
    start_time_sec: int,
    heuristic_func: Callable[[Graph, str, str], float | int],
    heuristic_multiplier: int,
    change_line_cost: int,
) -> tuple[list[Edge] | None, int, int]:
    # Priority queue (cost, start_stop, last_line, path, n_lines)
    queue = [
        (
            start_time_sec + heuristic_func(graph, start_stop, end_stop),
            start_time_sec, start_stop, None, [], 0
        )
    ]

    # The best score for each stop
    best_arrival_costs = {stop: float('inf') for stop in graph.nodes}
    best_arrival_costs[start_stop] = start_time_sec + heuristic_func(graph, start_stop, end_stop)
    visited = set()

    while queue:
        # Pop the stop with the earliest arrival time
        score, current_time, current_stop, last_line, path, n_lines = heapq.heappop(queue)

        # Check if we have already visited this stop
        if current_stop in visited:
            continue
        visited.add(current_stop)

        # End algorithm if we are at the destination
        if current_stop == end_stop:
            return path, score, n_lines

        # Get current node and iterate over all outgoing edges
        current_node = graph.nodes[current_stop]
        for edge in current_node.outgoing_edges:

            # Add one day if we are after midnight
            arrival_time = edge.arrival_sec + ((current_time // SECONDS_IN_DAY) * SECONDS_IN_DAY)

            # We cannot go earlier than we arrived at current stop
            if edge.departure_sec < current_time % SECONDS_IN_DAY:
                arrival_time = arrival_time + SECONDS_IN_DAY

            # If departure time is after arrival time, we know it's midnight case
            if edge.departure_sec > edge.arrival_sec:
                arrival_time = arrival_time + SECONDS_IN_DAY

            # Update the number of routes if we changed the line
            n_lines_new = n_lines
            if last_line != edge.line or current_time % SECONDS_IN_DAY != edge.departure_sec:
                n_lines_new += 1

            # Calculate the cost
            cost = (
                arrival_time
                + heuristic_func(graph, edge.end_stop_name, end_stop)
                * heuristic_multiplier
                + (n_lines_new - 1) * change_line_cost
            )

            # Update best time and push queue if we found a better path
            if cost < best_arrival_costs[edge.end_stop_name]:
                best_arrival_costs[edge.end_stop_name] = cost
                new_path = path + [edge]
                heapq.heappush(
                    queue,
                    (cost, arrival_time, edge.end_stop_name, edge.line, new_path, n_lines_new)
                )

    return None, -1, -1  # No path found


### Helper functions

In [13]:
def format_time(seconds) -> str:
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"


def convert_to_seconds(time: str):
    return (
        int(time.split(":")[0]) * 3600 +
        int(time.split(":")[1]) * 60
    )

### Shortest path CLI

In [25]:
CHANGE_LINE_COST = 10000000
HEURISTIC_MULTIPLIER = 1600

def shortest_path(
    graph: Graph,
    start_stop: str,
    end_stop: str,
    start_time_sec: int,
    criterion: str,
) -> tuple[list[Edge] | None, int, int]:
    if criterion == "t":
        return dijkstra_shortest_travel_time(
            graph=graph,
            start_stop=start_stop,
            end_stop=end_stop,
            start_time_sec=start_time_sec,
        )
    return astar_shortest_travel(
        graph=graph,
        start_stop=start_stop,
        end_stop=end_stop,
        start_time_sec=start_time_sec,
        heuristic_func=haversine_distance,
        heuristic_multiplier=HEURISTIC_MULTIPLIER,
        change_line_cost=CHANGE_LINE_COST,
    )

def main():
    if os.path.exists('./data/graph.pkl'):
        with open('./data/graph.pkl', 'rb') as f:
            g = pickle.load(f)
    else:
        df_ = pd.read_csv("./data/wroclaw-mpk.csv", low_memory=False)
        g = Graph.create_from_df(df_)

        with open('./data/graph.pkl', 'wb') as f:
            pickle.dump(g, f)

    # Dane testowe: małopanewska, hala stulecia
    start = input("Podaj przystanek początkowy: ").strip().lower()
    end = input("Podaj przystanek końcowy: ").strip().lower()
    criterion = input("Podaj kryterium: t/p (czas/przesiadki): ").strip().lower()
    start_time = input("Podaj czas początkowy (HH:MM): ")

    start_time_sec = convert_to_seconds(start_time)
    result = shortest_path(
        graph=g,
        start_stop=start,
        end_stop=end,
        start_time_sec=start_time_sec,
        criterion=criterion,
    )

    path, score, n_lines = result

    if path:
        print("Znaleziono ścieżkę:")
        for edge in path:
            print(
                f"{edge.start_stop_name} -> {edge.end_stop_name}, "
                f"linia {edge.line}, "
                f"{format_time(edge.departure_sec)} -> {format_time(edge.arrival_sec)}"
            )
        print(f"Czas dotarcia: {format_time(path[-1].arrival_sec)}")
        print(f"Score: {score}")
        print(f"Liczba przejazdow: {n_lines}")
    else:
        print("Brak połączenia")


if __name__ == "__main__":
    main()

Znaleziono ścieżkę:
małopanewska -> niedźwiedzia, linia 21, 08:00 -> 08:02
niedźwiedzia -> wrocław mikołajów (zachodnia), linia 3, 08:02 -> 08:04
wrocław mikołajów (zachodnia) -> pl. strzegomski (muzeum współczesne), linia 3, 08:04 -> 08:06
pl. strzegomski (muzeum współczesne) -> młodych techników, linia 3, 08:06 -> 08:07
młodych techników -> pl. jana pawła ii, linia 3, 08:07 -> 08:09
pl. jana pawła ii -> rynek, linia 3, 08:09 -> 08:11
rynek -> zamkowa, linia 3, 08:11 -> 08:12
zamkowa -> świdnicka, linia 3, 08:12 -> 08:14
świdnicka -> galeria dominikańska, linia 3, 08:14 -> 08:16
galeria dominikańska -> urząd wojewódzki (impart), linia d, 08:18 -> 08:21
urząd wojewódzki (impart) -> most grunwaldzki, linia d, 08:21 -> 08:22
most grunwaldzki -> pl. grunwaldzki, linia d, 08:22 -> 08:24
pl. grunwaldzki -> kliniki - politechnika wrocławska, linia 19, 08:25 -> 08:27
kliniki - politechnika wrocławska -> hala stulecia, linia 146, 08:27 -> 08:28
Czas dotarcia: 08:28
Score: 30480.0
Liczba przeja

## Traveling Salesman Problem

### Solution class
For representing tabu search solution variant.

In [19]:
class Solution:
    def __init__(self, stops_sequence: list[str]):
        self.stops = stops_sequence.copy()
        self.paths: list[tuple[int, list[Edge]]] = []
        self.cost = float('inf')

    @property
    def flat_path(self):
        return [edge for _, path in self.paths for edge in path]

    def duplicate(self) -> 'Solution':
        new = Solution(self.stops)
        new.paths = deepcopy(self.paths)
        new.cost = self.cost
        return new

    def calculate_cost(self, graph: Graph, time: int, criterion: str):
        self.paths = []
        self.cost = 0

        for i in range(len(self.stops) - 1):
            start = self.stops[i]
            end = self.stops[i+1]

            path, score, n_lines = shortest_path(
                graph=graph,
                start_stop=start,
                end_stop=end,
                start_time_sec=time,
                criterion=criterion,
            )
            if score is None:
                self.cost = float('inf')
                return

            self.paths.append((score, path))
            self.cost += score
            if path:
                time = path[-1].arrival_sec
            elif criterion == 't':
                time += score

### Helper functions

In [20]:
def sample(seq: Sequence, sample_size: int, strategy: str = 'random'):
    if strategy == 'random':
        return random.sample(seq, sample_size)
    elif strategy == 'systematic':
        length = len(seq)
        step = max(1, length // sample_size)
        return [seq[i] for i in range(0, length, step)][:sample_size]

    raise ValueError(f"Unknown sampling strategy: {strategy}")


def swap_segment(stops: list[Edge], start_idx: int, end_idx: int):
    new_stops = stops.copy()
    return (
        new_stops if start_idx >= end_idx
        else new_stops[:start_idx] + new_stops[start_idx:end_idx+1][::-1] + new_stops[end_idx+1:]
    )


def generate_neighbors(solution: Solution):
    stops = solution.stops
    total = len(stops)
    neighbors = []
    for start_idx in range(1, total - 2):
        for end_idx in range(start_idx + 1, total - 1):
            new_stops = swap_segment(stops, start_idx, end_idx)
            neighbors.append((start_idx, end_idx, new_stops))
    return neighbors

### Core calculation of Tabu Search

In [21]:
def tabu_search(
    graph: Graph,
    initial_solution: Solution,
    start_time_sec: int,
    criterion: str = 't',
    tabu_size_limited: bool = False,
    is_aspirational: bool = False,
    max_iterations: int = 100,
    sample_size: int = None,
    sample_strategy: str = 'random',
) -> Solution:
    current_solution = initial_solution
    current_solution.calculate_cost(graph, start_time_sec, criterion)
    best_solution = current_solution.duplicate()
    best_cost = current_solution.cost
    tabu_limit = max(5, len(current_solution.stops) // 2) if tabu_size_limited else None
    tabu_list = []

    for _ in range(max_iterations):

        # Generate all neighbors possible moves and optional sample them to reduce the search space
        neighbors = generate_neighbors(current_solution)
        if sample_size:
            neighbors = sample(neighbors, sample_size, sample_strategy)
        iteration_best_cost = float('inf')
        iteration_best_solution = None
        chosen_move = None

        for start, end, new_stops in neighbors:
            is_forbidden = ((start, end) in tabu_list) or ((end, start) in tabu_list)
            temp_solution = current_solution.duplicate()
            temp_solution.stops = new_stops
            temp_solution.calculate_cost(graph, start_time_sec, criterion)
            cost = temp_solution.cost
            if cost == float('inf'):
                continue

            # Skip forbidden except aspirational bypass
            if is_forbidden and not (is_aspirational and cost < best_cost):
                continue

            # Update the best solution if the cost is lower than the current best
            if cost < iteration_best_cost:
                iteration_best_cost = cost
                iteration_best_solution = temp_solution
                chosen_move = (start, end)

        if iteration_best_solution is None:
            break

        # Update the best solution if the cost is lower than the current best
        current_solution = iteration_best_solution
        if iteration_best_cost < best_cost:
            best_cost = iteration_best_cost
            best_solution = current_solution.duplicate()

        # Add the move to the tabu list and control its length
        tabu_list.append(chosen_move)
        if tabu_limit is not None and len(tabu_list) > tabu_limit:
            tabu_list.pop(0)  # Remove the oldest move

    return best_solution

### Tabu Search CLI

In [26]:
def main():
    if os.path.exists('./data/graph.pkl'):
        with open('./data/graph.pkl', 'rb') as f:
            g = pickle.load(f)
    else:
        df_ = pd.read_csv("./data/wroclaw-mpk.csv", low_memory=False)
        g = Graph.create_from_df(df_)

        with open('./data/graph.pkl', 'wb') as f:
            pickle.dump(g, f)

    # Dane testowe
    # małopanewska
    # Hala Stulecia;Stanki;RACŁAWICKA;Bałtycka;Wyszyńskiego;Stadion Olimpijski
    start_stop = input("Podaj przystanek początkowy: ").strip().lower()
    stops_to_visit_str = input("Podaj przystanki przejściowe rozdzielone średnikiem: ").strip()
    stops_to_visit = [s.lower() for s in stops_to_visit_str.split(';')]
    criterion = input("Podaj kryterium: t/p (czas/przesiadki): ").strip().lower()
    start_time = input("Podaj czas początkowy (HH:MM): ").strip()

    start_time_sec = convert_to_seconds(start_time)
    initial_stops = [start_stop] + stops_to_visit + [start_stop]
    initial_solution = Solution(initial_stops)

    solution = tabu_search(
        g,
        initial_solution,
        start_time_sec=start_time_sec,
        criterion=criterion,
        tabu_size_limited=False,
        is_aspirational=True,
        max_iterations=300,
        sample_size=None,
        sample_strategy='random',
    )

    for edge in solution.flat_path:
        print(
            f"{edge.start_stop_name} -> {edge.end_stop_name}, "
            f"linia {edge.line}, "
            f"{format_time(edge.departure_sec)} -> {format_time(edge.arrival_sec)}"
        )

    print("Całkowity koszt:", solution.cost)


if __name__ == "__main__":
    main()


małopanewska -> kwiska, linia 21, 08:00 -> 08:01
kwiska -> kolista, linia 21, 08:01 -> 08:05
kolista -> wejherowska (hala orbita), linia 18, 08:05 -> 08:08
wejherowska (hala orbita) -> milenijna (hala orbita), linia 119, 08:09 -> 08:11
milenijna (hala orbita) -> most milenijny, linia 119, 08:11 -> 08:12
most milenijny -> osobowicka (cmentarz), linia 119, 08:12 -> 08:14
osobowicka (cmentarz) -> osobowicka (cmentarz ii), linia 119, 08:14 -> 08:15
osobowicka (cmentarz ii) -> łużycka, linia 119, 08:15 -> 08:17
łużycka -> różanka, linia 119, 08:17 -> 08:18
różanka -> bezpieczna, linia 119, 08:18 -> 08:19
bezpieczna -> bałtycka (szkoła), linia 119, 08:19 -> 08:21
bałtycka (szkoła) -> bałtycka, linia 105, 08:21 -> 08:22
bałtycka -> broniewskiego, linia 105, 08:22 -> 08:24
broniewskiego -> trzebnicka, linia 144, 08:24 -> 08:27
trzebnicka -> dworzec nadodrze, linia 144, 08:27 -> 08:29
dworzec nadodrze -> słowiańska, linia 1, 08:29 -> 08:31
słowiańska -> nowowiejska, linia 1, 08:31 -> 08:33
nowo